In [ ]:
# ---- Install packages (run once per project, then comment out) ----
# !pip install openai pandas numpy matplotlib seaborn pyreadr tiktoken
# !pip install python-dotenv
#
# After installing, restart the kernel, then run all cells top-to-bottom.
# openai    — official Python SDK for the OpenAI API (GPT-4o, GPT-3.5, etc.)
# tiktoken  — OpenAI's tokenizer: lets us count tokens before sending to the API
# pyreadr   — reads .rds files directly from R (no need to export to CSV first)
# python-dotenv — optional: load API keys from a .env file instead of a text file

In [ ]:
# ============================================================
# 18 — Generative AI Concepts
# ============================================================
# Purpose : Teach how large language models (LLMs) work — what tokens
#           are, how text generation works step-by-step, what temperature
#           controls, and how to write effective clinical prompts. Uses the
#           OpenAI API (GPT-4o-mini) to demonstrate live generation.
#           No model training happens here — we are USERS of a trained LLM.
# Kernel  : .venv (Python 3.10+)
# Prereqs : openai_key.txt in project root (one line, just the API key)
# ============================================================

import os
import json
import time
import textwrap

import pandas as pd           # DataFrames — like R's tibble
import numpy as np            # Numerical arrays — like R's vectors
import matplotlib.pyplot as plt   # Plotting — like R's base graphics
import matplotlib.gridspec as gridspec
import seaborn as sns         # Statistical plots — like ggplot2
import tiktoken               # OpenAI's tokenizer — counts tokens before sending
from openai import OpenAI     # Official OpenAI Python SDK

# ---- Color palette (matches R scripts throughout the project) ----
BLUE  = "#2196F3"   # Negative / no disease
RED   = "#F44336"   # Positive / disease
GREEN = "#4CAF50"   # Correct / good
AMBER = "#FF9800"   # Warning / boundary
PURPLE = "#9C27B0"  # Accent

# ---- Plot style ----
plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "figure.facecolor": "white",
})

# ---- Output directory ----
OUT_DIR = "Outputs/Generative_AI"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- Model to use ----
# GPT-4o-mini: fast, cheap, capable — ideal for teaching demos.
# Swap to "gpt-4o" for higher quality if you want (costs more per token).
MODEL = "gpt-4o-mini"

# ---- API key loading ----
# The key is stored in openai_key.txt in your project root.
# IMPORTANT: Add openai_key.txt to .gitignore so it never goes to GitHub.

def load_openai_key(path: str = "openai_key.txt") -> str:
    """Load OpenAI API key from a plain-text file in the project folder."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            key = f.read().strip()
        if not key:
            raise ValueError("openai_key.txt is empty. Paste your API key into it.")
        return key
    except FileNotFoundError:
        raise FileNotFoundError(
            f"API key file not found: '{path}'\n"
            "Fix: Create openai_key.txt in your project root and paste your OpenAI API key as a single line."
        )

OPENAI_API_KEY = load_openai_key()
client = OpenAI(api_key=OPENAI_API_KEY)
print(f"API key loaded. Using model: {MODEL}")
print(f"Output directory: {OUT_DIR}")

In [ ]:
# ============================================================
# CELL 3 — What Is Generative AI? + Load Data + Build Clinical Notes
# ============================================================
#
# ---- WHAT IS GENERATIVE AI? ----
#
# Everything you have built in this curriculum so far has been DISCRIMINATIVE:
# given a patient's features, predict a category (heart disease: yes/no) or
# a number (healthcare cost). The model DISCRIMINATES between outcomes.
#
# Generative AI does something fundamentally different: it learns to PRODUCE
# new data that looks like the data it was trained on. A generative model
# trained on clinical text can write clinical notes. One trained on code
# can write code. One trained on the internet can answer questions.
#
# GPT (Generative Pre-trained Transformer) is the dominant family of
# generative AI for text. Here is what makes it different:
#
#   DISCRIMINATIVE ML (Scripts 01–15):
#     Input: patient features → Output: prediction (0/1 or a number)
#     Trained on: labeled examples (heart_disease = 1 or 0)
#     Can do: classify, regress, rank
#     Cannot do: write, explain, reason, create
#
#   LARGE LANGUAGE MODELS (This script):
#     Input: text (your prompt) → Output: text (the completion)
#     Trained on: hundreds of billions of words from the internet, books,
#                 code, scientific papers — with no labels
#     Can do: write, summarize, reason, translate, answer questions,
#             extract structured data, classify with instructions
#     Cannot do: guarantee accuracy; they can hallucinate confidently
#
# The key architectural innovation is the TRANSFORMER (2017, Google "Attention
# Is All You Need" paper). Transformers process entire sequences at once and
# learn which parts of the input to pay attention to — this is the 'attention
# mechanism'. Before transformers, models processed text word by word and
# forgot long-range context. Transformers hold the whole context window at once.
#
# GPT models are trained in two phases:
#   Phase 1 — Pre-training: predict the next token, billions of times, on
#             massive text corpora. This builds general language understanding.
#   Phase 2 — Fine-tuning / RLHF: human trainers rate responses; the model
#             learns to be helpful, harmless, and honest (not just fluent).
#
# When you call the API, you are not training anything. You are sending a
# PROMPT to an already-trained model and getting a COMPLETION back.
# The model has ~175 billion parameters (GPT-3) or more — these are fixed.
# What you control is the PROMPT and a few generation settings (temperature,
# max_tokens). Think of it as operating a very sophisticated calculator:
# the internals are fixed, your job is to ask the right question.
#
# ---- WHAT WE WILL DO IN THIS NOTEBOOK ----
#
# 1. Understand TOKENS — the atomic units LLMs operate on
# 2. Understand TEMPERATURE — how randomness is controlled
# 3. Understand CONTEXT WINDOW — the model's working memory
# 4. Demonstrate PROMPT ENGINEERING patterns:
#       zero-shot, few-shot, chain-of-thought, system prompts
# 5. Apply these patterns to our clinical dataset: summarize patients,
#    extract risk factors, classify risk level from a clinical note
# 6. Understand fine-tuning conceptually (we won't do it — it costs thousands
#    of dollars — but you'll know when and why to use it)
#

# ---- Load the clinical dataset ----
# We load our synthetic population health dataset (25,000 patients).
# We will select a small sample of patients and generate clinical notes
# from their structured data — this is a realistic workflow: you might
# have structured EHR fields and want the LLM to write a narrative summary.

try:
    import pyreadr
    result = pyreadr.read_r("Data/ml_teaching_data.rds")
    ml_data = result[None]
    print("Loaded .rds file directly via pyreadr")
except Exception:
    ml_data = pd.read_csv("Data/ml_teaching_data.csv")
    print("Loaded CSV fallback")

print(f"Dataset: {ml_data.shape[0]:,} rows × {ml_data.shape[1]} columns")

# ---- Select representative patients for demonstrations ----
# We want a variety: young/old, with/without heart disease, different risk profiles.
# We pick specific patients by filtering, not random sampling, so the notebook
# produces the same output every run.

np.random.seed(42)

# Convert numeric columns that should be integers
for col in ["age", "medication_count", "hospitalizations", "heart_disease",
            "diabetes", "cancer", "on_medication", "mortality"]:
    if col in ml_data.columns:
        ml_data[col] = ml_data[col].astype(int)

# Sample: 3 heart disease positive, 3 negative — representing clinical variety
hd_pos = ml_data[ml_data["heart_disease"] == 1].sample(3, random_state=42)
hd_neg = ml_data[ml_data["heart_disease"] == 0].sample(3, random_state=42)
demo_patients = pd.concat([hd_pos, hd_neg]).reset_index(drop=True)

print(f"\nDemo patients selected: {len(demo_patients)}")
print(demo_patients[["id", "age", "sex", "smoking_status", "bmi", 
                       "sbp", "cholesterol", "heart_disease"]].to_string(index=False))

# ---- Build clinical note text from structured fields ----
# This is a TEMPLATE approach: take structured data and produce a natural
# language summary. In real EHRs this happens automatically from structured
# entries. Here we construct it programmatically to feed into the LLM.

def build_clinical_note(row) -> str:
    """Convert a structured patient record into a short clinical note string."""
    sex_str  = row["sex"] if isinstance(row["sex"], str) else str(row["sex"])
    smk      = row["smoking_status"] if isinstance(row["smoking_status"], str) else str(row["smoking_status"])
    alc      = row["alcohol_use"] if isinstance(row["alcohol_use"], str) else str(row["alcohol_use"])
    edu      = row["education"] if isinstance(row["education"], str) else str(row["education"])

    comorbidities = []
    if int(row["diabetes"]) == 1:     comorbidities.append("diabetes")
    if int(row["cancer"]) == 1:       comorbidities.append("cancer")
    if int(row["heart_disease"]) == 1: comorbidities.append("heart disease")
    comorbid_str = ", ".join(comorbidities) if comorbidities else "no major comorbidities"

    note = (
        f"Patient is a {row['age']}-year-old {sex_str.lower()} with a {smk.lower()} smoking history "
        f"and {alc.lower()} alcohol use. "
        f"BMI {row['bmi']:.1f}, BP {row['sbp']:.0f}/{row['dbp']:.0f} mmHg, "
        f"cholesterol {row['cholesterol']:.0f} mg/dL, fasting glucose {row['fasting_glucose']:.0f} mg/dL. "
        f"Physical activity: {row['physical_activity']:.1f} hrs/week. "
        f"Education: {edu}. Annual income: ${row['income']:,.0f}. "
        f"Currently on {row['medication_count']} medication(s). "
        f"Known conditions: {comorbid_str}."
    )
    return note

demo_patients["clinical_note"] = demo_patients.apply(build_clinical_note, axis=1)

print("\n---- Example clinical note (Patient 1) ----")
print(textwrap.fill(demo_patients["clinical_note"].iloc[0], width=90))

In [ ]:
# ============================================================
# CELL 4 — Tokens, Temperature, Context Window & Prompt Engineering
# ============================================================
#
# ---- PART A: TOKENS ----
#
# LLMs don't see words. They see TOKENS.
# A token is roughly 3–4 characters of English text — sometimes a whole word,
# sometimes a word fragment, sometimes punctuation.
#
# Examples:
#   "Hello"          → 1 token
#   "Hello, world!"  → 4 tokens  (Hello / , / world / !)
#   "myocardial"     → 3 tokens  (my / ocard / ial)
#   "heart disease"  → 2 tokens
#
# Why does this matter?
#   1. COST: OpenAI charges per 1,000 tokens (input + output). Knowing token
#      counts helps you estimate costs before sending.
#   2. CONTEXT LIMIT: GPT-4o has a 128,000-token context window. If your
#      prompt + conversation history exceeds this, earlier content is dropped.
#      For clinical notes, a 25,000-patient dataset would NOT fit — you need
#      RAG or chunking (covered conceptually in Script 17).
#   3. SPEED: More tokens = slower response. Shorter prompts = faster API calls.
#
# tiktoken is OpenAI's open-source tokenizer — use it to count tokens
# before sending, so you know the cost and won't hit context limits.

# ---- Token counting demonstration ----
enc = tiktoken.encoding_for_model("gpt-4o")  # same tokenizer for gpt-4o-mini

token_examples = [
    "Hello, world!",
    "heart disease",
    "myocardial infarction",
    "Patient is a 67-year-old male smoker with elevated cholesterol.",
    demo_patients["clinical_note"].iloc[0],  # full note
]

print("---- Token counts for sample strings ----")
for text in token_examples:
    tokens = enc.encode(text)
    preview = text[:60] + ("..." if len(text) > 60 else "")
    print(f"  {len(tokens):>4} tokens | {preview}")

# Show all 6 clinical notes
total_note_tokens = sum(len(enc.encode(note)) for note in demo_patients["clinical_note"])
print(f"\nAll 6 demo clinical notes: {total_note_tokens} tokens total")
print(f"Context window (GPT-4o): 128,000 tokens — {total_note_tokens} tokens is {total_note_tokens/128000*100:.1f}% of the limit")

# ---- Cost estimate ----
# GPT-4o-mini pricing (as of early 2025): ~$0.15 per 1M input tokens,
# ~$0.60 per 1M output tokens. These prices change — check openai.com/pricing.
INPUT_COST_PER_1M  = 0.15   # $ per 1M input tokens
OUTPUT_COST_PER_1M = 0.60   # $ per 1M output tokens
est_cost = (total_note_tokens / 1_000_000) * INPUT_COST_PER_1M
print(f"Estimated input cost for these 6 notes: ${est_cost:.6f} (a fraction of a cent)")

print()

# ====================================================================
# ---- PART B: TEMPERATURE ----
# ====================================================================
#
# When an LLM generates the next token, it produces a probability distribution
# over all ~100,000 possible tokens. Temperature controls how peaked or flat
# that distribution is before sampling:
#
#   Temperature = 0.0:  Almost deterministic. The model always picks the
#                       highest-probability token. Good for factual tasks,
#                       classification, structured extraction.
#
#   Temperature = 0.7:  Balanced. Some randomness, generally coherent.
#                       Good for summaries, question answering.
#
#   Temperature = 1.5:  High randomness. Creative but may be incoherent.
#                       Good for brainstorming, poetry, fiction.
#
# The math: logits (raw scores) are divided by temperature before the
# softmax turns them into probabilities. Dividing by a small number (T < 1)
# sharpens the distribution (most probability mass on top token). Dividing
# by a large number (T > 1) flattens it (more tokens get non-trivial probability).
#
# Rule of thumb for clinical AI:
#   Structured extraction, classification → temperature = 0
#   Summaries, explanations              → temperature = 0.3–0.5
#   Creative generation, brainstorming   → temperature = 0.7–1.0
#   (Almost never go above 1.0 for clinical use)

# Visualize the temperature effect on a toy probability distribution
next_token_logits = np.array([3.0, 2.0, 1.5, 0.8, 0.3])  # raw scores for 5 candidate tokens
token_labels = ["hypertension", "elevated", "high", "increased", "abnormal"]
temperatures = [0.1, 0.7, 1.5]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
colors_temp = [BLUE, GREEN, RED]

for ax, T, color in zip(axes, temperatures, colors_temp):
    scaled_logits = next_token_logits / T
    probs = np.exp(scaled_logits) / np.exp(scaled_logits).sum()  # softmax
    bars = ax.bar(token_labels, probs, color=color, alpha=0.8, edgecolor="white")
    ax.set_title(f"Temperature = {T}", fontweight="bold")
    ax.set_ylabel("Probability" if T == 0.1 else "")
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis="x", rotation=30)
    for bar, p in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{p:.2f}", ha="center", va="bottom", fontsize=9)
    if T == 0.1:
        ax.set_xlabel("Almost always picks 'hypertension'")
    elif T == 0.7:
        ax.set_xlabel("Balanced: likely 'hypertension', sometimes others")
    else:
        ax.set_xlabel("Random: any token has a real chance")

fig.suptitle(
    "Temperature Controls Token Sampling\n"
    "Lower T → deterministic (good for clinical tasks). Higher T → creative (risky for facts).",
    fontsize=12, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "18_temperature_effect.png"), dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: 18_temperature_effect.png")

print()

# ====================================================================
# ---- PART C: CONTEXT WINDOW ----
# ====================================================================
#
# The context window is the model's working memory — the total number of tokens
# it can see at once (your prompt + the conversation history + its response).
#
# GPT-4o context window: 128,000 tokens ≈ ~100,000 words ≈ a short novel.
# This sounds large, but in production clinical AI you can exhaust it quickly:
#   - 100 clinical notes × 80 tokens each = 8,000 tokens (fine)
#   - Full 25,000-patient dataset          = ~2,000,000 tokens (WAY over limit)
#
# Solutions when data exceeds the context window:
#   1. RAG (Retrieval-Augmented Generation): retrieve only the relevant
#      documents using semantic search (Script 17), inject just those.
#   2. Chunking: break a long document into overlapping pieces, process each.
#   3. Summarization: compress long context into shorter form first.
#   4. Fine-tuning: bake the knowledge into the model weights directly
#      (expensive, but the knowledge is then always available).

# Show a visual of context window sizes for common models
model_context = pd.DataFrame({
    "Model": ["GPT-3.5", "GPT-4", "GPT-4o", "GPT-4o-mini", "Claude 3.5", "Gemini 1.5 Pro"],
    "Context (K tokens)": [16, 8, 128, 128, 200, 1000],
    "Family": ["OpenAI", "OpenAI", "OpenAI", "OpenAI", "Anthropic", "Google"]
})

fig, ax = plt.subplots(figsize=(10, 4))
family_colors = {"OpenAI": BLUE, "Anthropic": RED, "Google": GREEN}
bar_colors = [family_colors[f] for f in model_context["Family"]]
bars = ax.barh(model_context["Model"], model_context["Context (K tokens)"],
               color=bar_colors, alpha=0.85, edgecolor="white")
for bar, val in zip(bars, model_context["Context (K tokens)"]):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2,
            f"{val}K tokens", va="center", fontsize=10)

# Annotate our demo notes
ax.axvline(total_note_tokens / 1000, color=AMBER, linestyle="--", linewidth=1.5)
ax.text(total_note_tokens/1000 + 5, -0.4, f"Our 6 notes\n({total_note_tokens} tokens)",
        fontsize=9, color=AMBER)

ax.set_xlabel("Context Window (thousands of tokens)")
ax.set_title("LLM Context Windows\nAll modern models easily hold our 6 clinical notes",
             fontweight="bold")

# Color legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=f) for f, c in family_colors.items()]
ax.legend(handles=legend_elements, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "18_context_windows.png"), dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: 18_context_windows.png")

In [ ]:
# ============================================================
# CELL 5 — Prompt Engineering Patterns + Live API Demonstrations
# ============================================================
#
# ---- WHAT IS PROMPT ENGINEERING? ----
#
# A prompt is the text you send to the model. Prompt engineering is the
# craft of writing prompts that reliably produce the output you want.
# It is not magic — it is structured communication.
#
# Why does the prompt matter so much? Because the model is a text predictor.
# It predicts what text would plausibly follow your prompt. If you phrase
# your prompt like a good doctor asking a question, the model predicts text
# that a good doctor would write. If you phrase it poorly, it predicts
# something less useful.
#
# The four most important patterns:
#
#   1. SYSTEM PROMPT: Set the model's persona and constraints BEFORE the user
#      message. "You are a clinical AI assistant. Be concise. Never guess."
#      This is the most powerful lever — it shapes ALL subsequent responses.
#
#   2. ZERO-SHOT: Just ask. No examples. "Classify this note as high/low risk."
#      Works well for clear tasks; may drift for ambiguous ones.
#
#   3. FEW-SHOT: Provide 2–3 labeled examples BEFORE the actual question.
#      "Here are two examples of how I want you to classify notes: [examples].
#      Now classify this one." Dramatically improves consistency and format.
#
#   4. CHAIN-OF-THOUGHT (CoT): Ask the model to reason step by step before
#      answering. "Think through the risk factors one by one, then conclude."
#      This reduces errors on multi-step reasoning tasks — the model's
#      intermediate steps act like scratch paper.
#
# We will demonstrate all four on our clinical notes.
#

# ---- Helper: call the API and print formatted output ----

def call_llm(
    user_message: str,
    system_message: str = "You are a helpful clinical AI assistant.",
    temperature: float = 0.3,
    max_tokens: int = 400,
    label: str = ""
) -> str:
    """
    Send a prompt to the OpenAI API and return the response text.
    Prints formatted output including token usage.
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user",   "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    text = response.choices[0].message.content.strip()
    usage = response.usage
    if label:
        print(f"\n{'='*60}")
        print(f" {label}")
        print(f"{'='*60}")
    print(f"[Tokens: {usage.prompt_tokens} input + {usage.completion_tokens} output = {usage.total_tokens} total]")
    print(textwrap.fill(text, width=90))
    return text


# ---- Select one patient for the demonstrations ----
# We use a heart-disease-positive patient (first in our sorted sample)
# so the demonstrations show meaningful risk factor extraction.
demo_note = demo_patients.sort_values("heart_disease", ascending=False)["clinical_note"].iloc[0]
demo_age  = demo_patients.sort_values("heart_disease", ascending=False)["age"].iloc[0]
print("=== DEMONSTRATION PATIENT NOTE ===")
print(textwrap.fill(demo_note, width=90))
print()


# ================================================================
# DEMO 1 — Zero-Shot Classification
# ================================================================
# Zero-shot means: no examples. Just describe the task and ask.
# This works well when the task is unambiguous and the model has
# general clinical knowledge from pre-training.

zero_shot_prompt = f"""The following is a clinical note for a patient.
Classify this patient's heart disease risk as HIGH or LOW.
Give only the label and a one-sentence reason.

Clinical note:
{demo_note}"""

zero_shot_response = call_llm(
    user_message=zero_shot_prompt,
    system_message="You are a clinical risk assessment AI. Be concise and factual.",
    temperature=0.0,   # deterministic — we want consistent classification
    label="DEMO 1: Zero-Shot Classification (temperature=0)"
)
time.sleep(1)  # respect API rate limits


# ================================================================
# DEMO 2 — Few-Shot Classification
# ================================================================
# Few-shot means: provide labeled examples before the real question.
# This teaches the model your desired OUTPUT FORMAT, not just the task.
# The model sees the pattern in the examples and continues it.
#
# Rule of thumb: 2–5 examples. More than 5 rarely helps and costs more tokens.
# Make sure your examples cover both classes.

few_shot_prompt = f"""You classify clinical notes as HIGH or LOW heart disease risk.
Format: RISK: [HIGH/LOW] | KEY FACTORS: [comma-separated factors]

EXAMPLE 1:
Note: 72-year-old male, current smoker, BMI 31, BP 158/92, cholesterol 280, diabetes.
RISK: HIGH | KEY FACTORS: age, smoking, hypertension, high cholesterol, diabetes

EXAMPLE 2:
Note: 34-year-old female, never smoker, BMI 22, BP 110/70, cholesterol 160, no comorbidities.
RISK: LOW | KEY FACTORS: young age, non-smoker, normal BMI, normal BP, normal cholesterol

Now classify this note using the same format:
{demo_note}"""

few_shot_response = call_llm(
    user_message=few_shot_prompt,
    system_message="You are a clinical risk assessment AI. Follow the exact output format shown.",
    temperature=0.0,
    label="DEMO 2: Few-Shot Classification (with output format examples)"
)
time.sleep(1)


# ================================================================
# DEMO 3 — Chain-of-Thought Reasoning
# ================================================================
# Chain-of-thought (CoT) prompting asks the model to reason step by step
# before giving its final answer. This is especially useful when:
#   - The task requires combining multiple factors (e.g., clinical scoring)
#   - You need to audit the model's reasoning (explainability requirement)
#   - The task is complex enough that a direct answer might skip key logic
#
# The key phrase: "Think through each risk factor one by one, THEN conclude."
# When the model writes out its reasoning, it effectively has "scratch paper"
# which reduces reasoning errors.

cot_prompt = f"""You are evaluating a patient's cardiovascular risk.
Think through each of the following risk factors one by one:
  1. Age and sex
  2. Smoking status
  3. Blood pressure
  4. Cholesterol
  5. BMI and diabetes status
  6. Medications and comorbidities

After evaluating each factor, state your overall risk assessment (HIGH/MODERATE/LOW)
and estimate the probability of heart disease (as a percentage).

Clinical note:
{demo_note}"""

cot_response = call_llm(
    user_message=cot_prompt,
    system_message="You are a cardiologist AI. Reason carefully and transparently.",
    temperature=0.2,   # small randomness — we want thoughtful but not robotic
    max_tokens=600,    # CoT needs more tokens to write its reasoning
    label="DEMO 3: Chain-of-Thought Reasoning"
)
time.sleep(1)


# ================================================================
# DEMO 4 — Structured JSON Extraction
# ================================================================
# A key clinical AI task: take unstructured text and extract structured data.
# This is called "information extraction" or "named entity recognition."
#
# The technique: ask for JSON output and specify the exact schema.
# With temperature=0, the model almost always follows the schema exactly.
# This makes the output machine-readable — you can parse it directly in Python.
#
# Real-world use case: extract medication names, dosages, diagnoses, and
# dates from discharge summaries into structured database fields.

json_prompt = f"""Extract structured information from this clinical note.
Return ONLY valid JSON — no explanations, no markdown, no backticks.

Required JSON schema:
{{
  "age": integer,
  "sex": "male" or "female",
  "smoking_status": "never" | "former" | "current",
  "bmi_category": "underweight" | "normal" | "overweight" | "obese",
  "hypertension": true or false,
  "diabetes": true or false,
  "key_risk_factors": [list of strings],
  "risk_level": "low" | "moderate" | "high"
}}

Clinical note:
{demo_note}"""

json_raw = call_llm(
    user_message=json_prompt,
    system_message="You are a clinical data extraction AI. Return only valid JSON.",
    temperature=0.0,   # always 0 for structured extraction — minimizes hallucination
    max_tokens=300,
    label="DEMO 4: Structured JSON Extraction"
)

# Parse the JSON response — if the model follows instructions, this should work
try:
    extracted = json.loads(json_raw)
    print("\nParsed JSON successfully:")
    for key, val in extracted.items():
        print(f"  {key}: {val}")
except json.JSONDecodeError as e:
    print(f"\nJSON parse failed (model didn't follow format exactly): {e}")
    print("Raw response was:", json_raw[:200])
time.sleep(1)


# ================================================================
# DEMO 5 — Patient Summary (Creative / Narrative)
# ================================================================
# Finally, a generative task: write a patient summary suitable for a
# handoff note. We increase temperature slightly to allow natural phrasing.
# This mirrors how LLMs are actually used in clinical documentation tools.
#
# Note: LLMs can hallucinate specific numbers (vitals, lab values).
# Always instruct the model to use ONLY information from the input note —
# never invent data. This is a critical safety guardrail for clinical AI.

# Run demo on all 6 patients (quick summaries)
summary_results = []

print("\n" + "="*60)
print(" DEMO 5: Patient Summary Generation (all 6 demo patients)")
print("="*60)

for _, row in demo_patients.iterrows():
    note = row["clinical_note"]
    hd_label = "[KNOWN: Heart Disease +]" if row["heart_disease"] == 1 else "[KNOWN: Heart Disease −]"

    summary_prompt = (
        "Write a 2-sentence clinical handoff summary for this patient. "
        "Use ONLY facts present in the note — do not add or invent any information. "
        "End with: 'Risk assessment: [HIGH/MODERATE/LOW].'\n\n"
        f"Clinical note:\n{note}"
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a clinical documentation AI. Be concise and factual. Never invent information."},
            {"role": "user",   "content": summary_prompt}
        ],
        temperature=0.4,
        max_tokens=150
    )
    summary = response.choices[0].message.content.strip()
    summary_results.append(summary)

    print(f"\nPatient {row['id']} | Age {row['age']} | {hd_label}")
    print(textwrap.fill(summary, width=90, initial_indent="  ", subsequent_indent="  "))
    time.sleep(0.5)  # small delay to stay within API rate limits

demo_patients["llm_summary"] = summary_results
demo_patients[["id", "age", "heart_disease", "llm_summary"]].to_csv(
    os.path.join(OUT_DIR, "18_patient_summaries.csv"), index=False
)
print(f"\nSaved all 6 summaries to {OUT_DIR}/18_patient_summaries.csv")

In [ ]:
# ============================================================
# CELL 6 — Prompt Engineering Comparison + Fine-Tuning + Summary
# ============================================================
#
# ---- VISUALIZE: PROMPT ENGINEERING QUALITY COMPARISON ----
#
# Different prompt strategies produce different output quality for the
# same underlying task. This visualization summarizes the tradeoffs.

prompt_comparison = pd.DataFrame({
    "Strategy": ["Zero-Shot", "Few-Shot", "Chain-of-Thought", "System Prompt", "JSON Extraction"],
    "Format Consistency": [60, 90, 70, 80, 98],
    "Reasoning Quality":  [65, 70, 95, 75, 50],
    "Token Cost (relative)": [1.0, 1.5, 2.5, 1.1, 1.2],
    "Best For": [
        "Simple, clear tasks",
        "Consistent formatting",
        "Complex multi-factor decisions",
        "Persona & safety constraints",
        "Structured data extraction"
    ]
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Format Consistency vs Reasoning Quality bubble chart
ax = axes[0]
scatter_colors = [BLUE, GREEN, RED, AMBER, PURPLE]
for i, row in prompt_comparison.iterrows():
    size = row["Token Cost (relative)"] * 300
    ax.scatter(row["Format Consistency"], row["Reasoning Quality"],
               s=size, color=scatter_colors[i], alpha=0.8, edgecolors="white", linewidths=1.5)
    ax.annotate(row["Strategy"],
                (row["Format Consistency"], row["Reasoning Quality"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)

ax.set_xlabel("Format Consistency (%)\nHow reliably output matches desired format")
ax.set_ylabel("Reasoning Quality (%)\nDepth and accuracy of reasoning")
ax.set_title("Prompt Strategy Tradeoffs\nBubble size = relative token cost",
             fontweight="bold")
ax.set_xlim(50, 105)
ax.set_ylim(40, 105)
ax.grid(True, alpha=0.3)
ax.axvline(80, color="grey", linestyle=":", alpha=0.5)
ax.axhline(75, color="grey", linestyle=":", alpha=0.5)
ax.text(102, 76, "Target\nzone", fontsize=8, color="grey", ha="right")

# Plot 2: When to use each strategy
ax2 = axes[1]
ax2.axis("off")

table_data = [
    ["Zero-Shot",         "Fast, cheap",          "Simple Q&A, triage"],
    ["Few-Shot",          "Format control",        "Structured reports"],
    ["Chain-of-Thought",  "Auditable reasoning",   "Risk scoring, dx"],
    ["System Prompt",     "Role + safety guardrails","Any clinical tool"],
    ["JSON Extraction",   "Machine-readable output","EHR integration"],
]

col_labels = ["Strategy", "Strength", "Clinical Use Case"]
tbl = ax2.table(
    cellText=table_data,
    colLabels=col_labels,
    loc="center",
    cellLoc="left"
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 2.0)

for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor("#2196F3")
    tbl[(0, j)].set_text_props(color="white", fontweight="bold")
for i in range(1, len(table_data) + 1):
    bg = "#f5f5f5" if i % 2 == 0 else "white"
    for j in range(len(col_labels)):
        tbl[(i, j)].set_facecolor(bg)

ax2.set_title("Quick Reference: Prompt Engineering Strategies",
              fontweight="bold", pad=20)

plt.suptitle(
    "Prompt Engineering for Clinical AI\n"
    "Choose strategy based on task requirements, not habit.",
    fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "18_prompt_engineering_comparison.png"),
            dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: 18_prompt_engineering_comparison.png")


# ====================================================================
# ---- FINE-TUNING — CONCEPTUAL OVERVIEW (no code — too expensive) ----
# ====================================================================
#
# Fine-tuning means CONTINUING to train an existing LLM on your own data,
# so its weights are adjusted for your specific domain or task.
#
# ---- When fine-tuning makes sense ----
#
#   REASON                         | EXAMPLE
#   Specialized vocabulary         | Radiology reports, pathology notes
#   Consistent output format       | Always produce ICD-10 codes in column 3
#   Replace lengthy system prompts | Bake persona into model weights
#   Proprietary data the API       | Your hospital's discharge note style
#   model has never seen           |
#
# ---- When fine-tuning does NOT make sense ----
#
#   - You only have < 50 examples → few-shot prompting is cheaper and works
#   - You want the model to know NEW FACTS → RAG is better (fine-tuning
#     teaches STYLE and FORMAT, not factual knowledge reliably)
#   - Budget is limited → fine-tuning costs $10–$1000+ depending on data size
#
# ---- The fine-tuning process ----
#
#   1. Prepare training pairs: {"prompt": "...", "completion": "..."}  (JSONL format)
#      You need 50–500 high-quality examples minimum.
#
#   2. Upload to OpenAI: client.files.create(file=..., purpose="fine-tune")
#
#   3. Start fine-tune job: client.fine_tuning.jobs.create(
#          training_file=file_id, model="gpt-4o-mini")
#      This can run for 30 minutes to several hours.
#
#   4. Use your fine-tuned model by its new ID:
#      client.chat.completions.create(model="ft:gpt-4o-mini:your-org:your-model:xxxxx")
#
# ---- Comparison: Prompt Engineering vs Fine-Tuning vs RAG ----
#
#   APPROACH             | COST  | DATA NEEDED | BEST FOR
#   Prompt Engineering   | Free  | 0 examples  | Most tasks
#   Few-Shot Prompting   | Free  | 2–10 examples| Format/style consistency
#   Fine-Tuning          | $$$   | 50–500 examples| Specialized tasks at scale
#   RAG                  | $     | Any size    | Current knowledge, large docs
#
# Rule of thumb: try prompt engineering first. Only fine-tune if it fails
# at scale or you have strong format/style requirements.
#

# ====================================================================
# ---- HALLUCINATION: THE CORE RISK OF CLINICAL LLM USE ----
# ====================================================================
#
# Hallucination = the model generates text that sounds correct but is
# factually wrong. This happens because LLMs are trained to produce
# plausible text, not verified truth.
#
# Clinical risk examples:
#   - Model invents a drug interaction that doesn't exist
#   - Model confidently states a lab value that contradicts the input note
#   - Model references a guideline from the wrong year
#
# Mitigation strategies:
#   1. Retrieval-Augmented Generation (RAG): ground answers in source documents
#   2. Temperature=0 for factual tasks: reduces (but doesn't eliminate) drift
#   3. Instruct: "Use ONLY information from the note. If unsure, say unsure."
#   4. Structured output (JSON): easier to validate than free text
#   5. Human-in-the-loop: LLM drafts, human reviews — never fully autonomous
#      for high-stakes clinical decisions
#
# The bottom line: LLMs are powerful TOOLS, not autonomous agents. In clinical
# settings, treat them like a smart but occasionally-wrong junior colleague:
# useful for first drafts, not trusted for final decisions without review.

# ====================================================================
# ---- WHAT WE LEARNED ----
# ====================================================================
print("""
============================================================
 WHAT WE LEARNED — Script 18: Generative AI Concepts
============================================================

1. GENERATIVE vs DISCRIMINATIVE: Scripts 01–15 predict existing labels.
   LLMs generate new text — they don't classify; they complete.

2. TOKENS are the unit of LLM computation. Roughly 3–4 characters.
   Token count drives cost, speed, and context window limits.
   Always use tiktoken to estimate before sending large payloads.

3. TEMPERATURE controls randomness. 0 = deterministic (good for
   clinical classification/extraction). >0.7 = creative but risky
   for facts. Match temperature to task, not to preference.

4. CONTEXT WINDOW is working memory. GPT-4o holds 128K tokens —
   enough for dozens of notes, not thousands. Use RAG or chunking
   for large datasets.

5. PROMPT ENGINEERING shapes model behavior without changing weights:
   - System prompt: persona and constraints
   - Zero-shot: just ask
   - Few-shot: show examples to fix output format
   - Chain-of-thought: force step-by-step reasoning
   - JSON extraction: get machine-readable structured output

6. FINE-TUNING adjusts model weights on your data. Useful for style
   and format at scale. Expensive. Not a substitute for RAG when
   you need current facts.

7. HALLUCINATION is the primary clinical risk. Mitigate with:
   temperature=0, grounded prompts, structured output, human review.

HOW THIS FITS THE CURRICULUM:
  Scripts 01–15 (R): Discriminative ML — predict from structured features
  Script 16       : NLP — process text as features for ML
  Script 17       : Embeddings — represent meaning as vectors, enable RAG
  Script 18 (here): Generative AI — LLMs as text producers and reasoners
  Script 19       : Synthetic data — generative models for tabular data
  Script 20       : Agents — LLMs + tools + loops = autonomous workflows

WHAT PROBLEMS IS THIS BEST FOR?
  - Clinical documentation (note generation, summarization)
  - Structured extraction from free-text (medications, diagnoses, dates)
  - Patient-facing communication (discharge instructions, explanations)
  - Decision support drafts (always with human review)
  - Cohort building from narrative text where structured codes are absent

WHAT IT IS NOT FOR:
  - Predicting numerical outcomes — use the regression/ML models from Scripts 01–15
  - Fully autonomous clinical decisions — human oversight is required
  - Tasks requiring guaranteed accuracy — hallucination cannot be eliminated
============================================================
""")

# ---- Save demo patients with LLM outputs ----
demo_patients[["id", "age", "sex", "heart_disease", "clinical_note", "llm_summary"]].to_csv(
    os.path.join(OUT_DIR, "18_demo_patients_full.csv"), index=False
)
print(f"Full demo patient table saved to {OUT_DIR}/18_demo_patients_full.csv")
print(f"\nAll outputs saved to: {OUT_DIR}/")
print("  18_temperature_effect.png")
print("  18_context_windows.png")
print("  18_prompt_engineering_comparison.png")
print("  18_patient_summaries.csv")
print("  18_demo_patients_full.csv")